# 04 · Baseline and candidate-model comparison (cross-validation only)

Reads `reports/tables/cv_comparison.csv`, the ablation tables, the OOF parquet files, and the figures written by `python -m ssn train-cv`. Every pipeline was fitted inside 5-fold stratified CV on the training split; the held-out test split was not opened.

**No model is selected here.** Selection happens in Milestone 6 with a multi-criteria matrix that excludes accuracy.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from ssn.paths import repo_root

ROOT = repo_root()
T, F, P = ROOT / 'reports' / 'tables', ROOT / 'reports' / 'figures', ROOT / 'data' / 'processed'
cv = pd.read_csv(T / 'cv_comparison.csv')
print('variants:', len(cv), '| seed:', cv.seed.iloc[0], '| folds:', cv.n_folds.iloc[0], '| K:', cv.capacity_k.iloc[0], 'of', cv.n_cohort_expected.iloc[0])
cv[['variant', 'estimator', 'key_params', 'steps', 'fit_time_s_mean']]

## 1. Primary and ranking metrics (fold mean ± std)

In [ ]:
cols = ['variant', 'pr_auc_mean', 'pr_auc_std', 'roc_auc_mean', 'roc_auc_std', 'recall_at_k_mean', 'precision_at_k_mean', 'brier_mean', 'ece_mean']
cv[cols].round(4).sort_values('pr_auc_mean', ascending=False)

### Why Recall@K is near 0.17 for every model

K is the illustrative outreach capacity (50 students = 10 per week × 5 weeks) applied to a cohort of 885. Scaled to a validation fold of ~708 rows, K becomes ~40, while each fold holds ~227 dropouts. The maximum achievable Recall@K is therefore ~40/227 ≈ 0.176. Precision@K near 0.97 says the top-40 list is almost entirely true dropouts; Recall@K says capacity, not the model, is the binding constraint. Milestone 6 reports sensitivity over 2-, 5- and 10-week windows.

In [ ]:
# exact ceiling from the OOF file: positives per fold vs k per fold
oof = pd.read_parquet(P / 'oof_dummy__sel-none__imb-none.parquet')
pos_per_fold = oof.groupby('fold').y_true.sum()
rows_per_fold = oof.groupby('fold').size()
k_fold = cv.k_fold_mean.iloc[0]
print('rows per fold:', rows_per_fold.tolist())
print('dropouts per fold:', pos_per_fold.tolist())
print('k per fold:', k_fold, '-> Recall@K ceiling per fold:', (k_fold / pos_per_fold).round(4).tolist())
cv[['variant', 'k_fold_mean', 'recall_at_k_mean', 'precision_at_k_mean', 'recall_at_k2w_mean', 'recall_at_k10w_mean']].round(4)

## 2. Threshold metrics at 0.5 and confusion matrices

Reported for completeness. The deployed threshold is chosen in Milestone 6 from OOF scores, so these 0.5-based values are not the operating point.

In [ ]:
cv[['variant', 'precision_at_threshold_mean', 'recall_at_threshold_mean', 'f1_at_threshold_mean', 'accuracy_at_threshold_mean']].round(4)

In [ ]:
pd.read_csv(T / 'cv_confusion.csv')

## 3. Out-of-fold PR curves and calibration

In [ ]:
display(Image(str(F / 'cv_pr_curves.png')))
display(Image(str(F / 'cv_calibration.png')))

## 4. Imbalance treatment: class weighting vs SMOTENC (research R-07 rule)

In [ ]:
imb = cv[cv.selection == 'none'].pivot(index='model', columns='imbalance', values='pr_auc_mean').round(4)
imb['delta_smotenc_minus_cw'] = (imb.get('smotenc') - imb.get('class_weight')).round(4)
imb

## 5. Feature selection setting (k = 30, embedded L1) vs all features

In [ ]:
sel = cv[cv.imbalance != 'smotenc'].pivot(index='model', columns='selection', values='pr_auc_mean').round(4)
sel['delta_decided_minus_none'] = (sel.get('decided') - sel.get('none')).round(4)
sel

## 6. Ablations (analysis only; promoted columns are never allowed for the deployed model)

In [ ]:
display(pd.read_csv(T / 'ablation_ambiguous.csv').round(4))
pd.read_csv(T / 'ablation_sensitive.csv').round(4)

## 7. OOF files for Milestone 6

One parquet per variant with `record_id`, `fold`, `y_true`, `score`; the threshold and bands are derived from these, never from test data.

In [ ]:
sorted(p.name for p in P.glob('oof_*.parquet'))